### Chargement & préparation de base

In [ ]:
# --- Load aggregated per-quote dataset (from EDA step) ---
from pathlib import Path
import pandas as pd
import numpy as np

CANDIDATES = ["../data/quotes_grouped_mysql.json", "../data/quotes_grouped.json"]
DATA = next((p for p in CANDIDATES if Path(p).exists()), None)
assert DATA is not None, "Aggregated dataset not found. Run the EDA aggregation first."

df = pd.read_json(DATA)
df = df[df["proposed_price_accepted"].notnull()].copy()

# Keep only columns used by the model
FEATURES = ["source_language","target_language","unit","number_of_unit","document_type","requested_deadline"]
TARGET = "proposed_price_accepted"

# Basic cleaning/casting
for col in ["source_language","target_language","unit","document_type"]:
    if col in df.columns:
        df[col] = df[col].astype("category")
df["number_of_unit"] = pd.to_numeric(df["number_of_unit"], errors="coerce")
df["requested_deadline"] = pd.to_numeric(df["requested_deadline"], errors="coerce")

df = df.dropna(subset=[TARGET])  # target must exist
X = df[FEATURES].copy()
y = df[TARGET].astype(float).copy()

print(X.shape, y.shape)
X.head()

### Imputation explicite des NaN numériques

In [ ]:
# Make missing values explicit for numeric features (so effects are visible in EBM)
for col in ["number_of_unit", "requested_deadline"]:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")
        X[col] = X[col].fillna(-1000)  # sentinel visible dans les bins EBM

### Split train/test

In [ ]:
# Time-based split if created_at exists, otherwise random split
from sklearn.model_selection import train_test_split

if "created_at" in df.columns:
    dfts = df.copy()
    dfts["created_at"] = pd.to_datetime(dfts["created_at"], errors="coerce")
    dfts = dfts.sort_values("created_at")
    split_idx = int(0.8 * len(dfts))
    train_idx = dfts.index[:split_idx]
    test_idx  = dfts.index[split_idx:]
    X_train, X_test = X.loc[train_idx], X.loc[test_idx]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

print("Train/Test:", X_train.shape, X_test.shape)

### Entraînement EBM

In [ ]:
from interpret.glassbox import ExplainableBoostingRegressor
from sklearn.metrics import mean_absolute_error

ebm = ExplainableBoostingRegressor(
    random_state=0,
    interactions=0,        # keep purely additive for readability
    max_bins=256,
    learning_rate=0.01,
    max_leaves=3,
)
ebm.fit(X_train, y_train)

y_pred = ebm.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))

### Courbe EBM en valeur absolue

In [ ]:
# Plot shape in absolute predicted price (not centered)
def plot_ebm_shape_absolute(model, X_ref, feature, filename, grid_size=200):
    import numpy as np, matplotlib.pyplot as plt, pandas as pd

    X0 = X_ref.copy()
    # baseline: modes/medians for other features
    baseline = {}
    for c in X0.columns:
        if str(X0[c].dtype) == "category" or X0[c].dtype == object:
            baseline[c] = X0[c].mode(dropna=True).iloc[0]
        else:
            baseline[c] = float(pd.to_numeric(X0[c], errors="coerce").median(skipna=True))

    # grid for the selected feature
    if str(X0[feature].dtype) == "category" or X0[feature].dtype == object:
        grid_vals = X0[feature].astype(str).value_counts().index.tolist()
    else:
        xnum = pd.to_numeric(X0[feature], errors="coerce")
        lo, hi = float(np.nanpercentile(xnum, 1)), float(np.nanpercentile(xnum, 99))
        grid_vals = np.linspace(lo, hi, grid_size)

    rows = []
    for v in grid_vals:
        row = baseline.copy()
        row[feature] = v
        rows.append(row)
    import pandas as pd
    Xg = pd.DataFrame(rows)[X0.columns]
    yp = model.predict(Xg)

    import matplotlib.pyplot as plt
    plt.figure(figsize=(7,4))
    plt.plot(grid_vals, yp, linewidth=2)
    # (Option) enlever le titre si tu veux un rendu “clean” dans la thèse
    # plt.title(f"EBM predicted price — varying {feature}")
    plt.xlabel(feature); plt.ylabel("Predicted price")
    plt.tight_layout(); plt.savefig(filename, dpi=180); plt.show()

plot_ebm_shape_absolute(ebm, X_train, "number_of_unit", "ebm_shape_number_of_unit_abs.png")

### Ancienne figure

In [ ]:
# Plot a shape function by probing the model on a grid (additive model → shape up to a constant)
import matplotlib.pyplot as plt

def plot_ebm_shape_grid(model, X_ref, feature, filename, grid_size=200):
    X0 = X_ref.copy()
    # build a baseline row with modes/medians
    baseline = {}
    for c in X0.columns:
        if str(X0[c].dtype) == "category" or X0[c].dtype == object:
            baseline[c] = X0[c].mode(dropna=True).iloc[0]
        else:
            baseline[c] = float(X0[c].median(skipna=True))
    # grid for the selected feature
    if str(X0[feature].dtype) == "category" or X0[feature].dtype == object:
        grid_vals = X0[feature].astype(str).value_counts().index.tolist()
    else:
        lo = float(np.nanpercentile(X0[feature], 1))
        hi = float(np.nanpercentile(X0[feature], 99))
        grid_vals = np.linspace(lo, hi, grid_size)

    rows = []
    for v in grid_vals:
        row = baseline.copy()
        row[feature] = v
        rows.append(row)
    Xg = pd.DataFrame(rows)[X0.columns]
    yp = model.predict(Xg)
    # center to show relative contribution shape
    yp_centered = yp - yp[0]

    plt.figure(figsize=(7,4))
    plt.plot(grid_vals, yp_centered, linewidth=2)
    plt.title(f"EBM shape — {feature}")
    plt.xlabel(feature); plt.ylabel("Relative contribution (centered)")
    plt.tight_layout(); plt.savefig(f"ebm_shape_{feature}.png", dpi=180); plt.show()

plot_ebm_shape_grid(ebm, X_train, "number_of_unit", "ebm_shape_number_of_unit.png")

### Importance avec noms corrects

In [ ]:
# Proper term names from term_features_ + feature_names_in_
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt

term_names = []
for feats in ebm.term_features_:
    if len(feats) == 1:
        term_names.append(ebm.feature_names_in_[feats[0]])
    else:
        term_names.append(" x ".join([ebm.feature_names_in_[i] for i in feats]))

importances = [float(np.mean(np.abs(ts))) for ts in ebm.term_scores_]
imp_df = pd.DataFrame({"feature": term_names, "importance": importances}) \
         .sort_values("importance", ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(data=imp_df, x="importance", y="feature")
plt.title("EBM — average absolute contribution by feature")
plt.xlabel("Average |contribution|"); plt.ylabel("")
plt.tight_layout(); plt.savefig("ebm_importances.png", dpi=180); plt.show()

imp_df.head(10)